# Image Sequence-Order Test Prediction Collection

This notebook exports aligned test-set predictions for the three final image-sequence ablation models:

- **Ordered LSTM** — chronological 30-frame sequences.
- **Shuffled LSTM** — the same 30 frames in a deterministic random order using `SEED + 2` for the test set.
- **Mean aggregation** — temporal mean of the 30 EfficientNetV2 frame-level feature vectors.

The output contains one row per image-view sequence (756 rows), with `reenactment_id` linking the Central- and Side-view sequences of the same reenactment. It is intended as the common input for the Ordered-vs.-Shuffled and Ordered-vs.-Mean significance-test notebooks.

## 1. Setup


### 1.0 Add log filter

Add the log filter below or tensorflow will print thousands of lines of uninformative log messages.  


In [1]:
import re
import ipykernel.iostream

TF_LOG_FILTER_PATTERNS = [
    r'ptx\d+.*is not a recognized feature for this target',
    r'is not a recognized feature for this target \(ignoring feature\)',
    r'\(ignoring feature\)',
    r'successful NUMA node read from SysFS had negative value \(-1\)',
    r'gpu_timer\.cc:114\] Skipping the delay kernel, measurement accuracy will be reduced',
]

KERAS_PROGRESS_PATTERNS = [
    r'ms/step',
    r's/step',
    r'ETA:',
    r'\d+/\d+ \[',   # 12/64 [===>...]
]

_original_write = ipykernel.iostream.OutStream.write

def _filtered_write(self, msg, *args, **kwargs):
    text = str(msg)

    if any(re.search(p, text) for p in KERAS_PROGRESS_PATTERNS):
        _original_write(self, text, *args, **kwargs)
        return

    buf = getattr(self, '_tf_log_filter_buf', '')
    buf += text

    if '\n' not in buf:
        setattr(self, '_tf_log_filter_buf', buf)
        return

    lines = buf.splitlines(keepends=True)
    if not buf.endswith('\n'):
        incomplete = lines.pop()
    else:
        incomplete = ''

    for line in lines:
        if any(re.search(p, line) for p in TF_LOG_FILTER_PATTERNS):
            continue
        _original_write(self, line, *args, **kwargs)

    setattr(self, '_tf_log_filter_buf', incomplete)

ipykernel.iostream.OutStream.write = _filtered_write

print('Notebook log filter installed (targeted, keeps Keras steps).')

Notebook log filter installed (targeted, keeps Keras steps).


### 1.1 Imports and paths

The dataset and ordered-model paths follow the existing `emohevrdb-dfer` Docker/repository conventions. Place the final optimized shuffled and mean models under the filenames below, or adjust only the corresponding path constants.

In [2]:
from pathlib import Path
import gc
import re

import ipykernel.iostream
import keras
import numpy as np
import pandas as pd
import tensorflow as tf


DATASET_ROOT = Path('/workspace/datasets')
MODEL_ROOT = Path('./models')

DI_TEST_PATH = DATASET_ROOT / 'emoji-hero-vr-db-di' / 'test_set'

ORDERED_MODEL_PATH = MODEL_ROOT / 'image_sequence_model.keras'
SHUFFLED_MODEL_PATH = MODEL_ROOT / 'image_sequence_shuffled_model.keras'
MEAN_MODEL_PATH = MODEL_ROOT / 'image_sequence_mean_model.keras'

SEED = 13
SEQUENCE_LENGTH = 30
IMAGE_SIZE = (224, 224, 3)
BATCH_SIZE = 32
TEST_SEQUENCE_SHUFFLE_SEED = SEED + 2

keras.mixed_precision.set_global_policy('mixed_float16')

print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

for path in [DI_TEST_PATH, ORDERED_MODEL_PATH, SHUFFLED_MODEL_PATH, MEAN_MODEL_PATH]:
    print(f"{path}: {'OK' if path.exists() else 'MISSING'}")

2026-09-18 16:58:57.376927: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-18 16:58:57.385795: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-18 16:58:57.388686: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow: 2.17.0
Keras: 3.12.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
/workspace/datasets/emoji-hero-vr-db-di/test_set: OK
models/image_sequence_model.keras: OK
models/image_sequence_shuffled_model.keras: OK
models/image_sequence_mean_model.keras: OK


### 1.3 Class mapping

All dynamic image models use the canonical EmoHeVRDB class order.

In [3]:
ID_TO_EMOTION = {
    0: 'Anger',
    1: 'Disgust',
    2: 'Fear',
    3: 'Happiness',
    4: 'Neutral',
    5: 'Sadness',
    6: 'Surprise'
}

EMOTION_TO_ID = {v: k for k, v in ID_TO_EMOTION.items()}
EMOTIONS = list(EMOTION_TO_ID.keys())

## 2. Parse and verify the image-sequence test set

Each sequence directory name has the form

`<timestamp>-<set-id>-<participant-id>-<level-id>-<emoji-id>-<emotion-id>-<camera-index>`.

The directory name is the unique view-level `sample_id`; removing the camera index yields the shared `reenactment_id`.

In [4]:
def parse_image_sequence_dir(sequence_dir: Path) -> dict:
    parts = sequence_dir.name.split('-')

    if len(parts) != 7:
        raise ValueError(f'Unexpected sequence directory name: {sequence_dir.name}')

    timestamp, set_id, participant_id, level_id, emoji_id, emotion_id, camera_index = parts
    reenactment_id = '-'.join(parts[:-1])

    image_paths = sorted(
        [path for path in sequence_dir.iterdir() if path.is_file()],
        key=lambda path: int(path.name.split('-')[0])
    )

    return {
        'sample_id': sequence_dir.name,
        'reenactment_id': reenactment_id,
        'timestamp': int(timestamp),
        'set_id': int(set_id),
        'participant_id': int(participant_id),
        'level_id': int(level_id),
        'emoji_id': int(emoji_id),
        'true_label_id': int(emotion_id),
        'camera_index': int(camera_index),
        'perspective': 'Central' if camera_index == '0' else 'Side',
        'image_sequence_path': str(sequence_dir),
        'image_paths': [str(path) for path in image_paths],
        'emotion_dir': sequence_dir.parent.name
    }


image_rows = []

for class_dir in sorted(DI_TEST_PATH.iterdir()):
    if not class_dir.is_dir():
        continue

    for sequence_dir in sorted(class_dir.iterdir()):
        if sequence_dir.is_dir():
            image_rows.append(parse_image_sequence_dir(sequence_dir))

prediction_df = pd.DataFrame(image_rows).reset_index(drop=True)
prediction_df['true_label'] = prediction_df['true_label_id'].map(ID_TO_EMOTION)

display(prediction_df.head())

,sample_id,reenactment_id,timestamp,set_id,participant_id,level_id,emoji_id,true_label_id,camera_index,perspective,image_sequence_path,image_paths,emotion_dir,true_label
0,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,0,Central,/workspace/datasets/emoji-hero-vr-db-di/test_s...,[/workspace/datasets/emoji-hero-vr-db-di/test_...,Anger,Anger
1,1700478995850-2-1-1-0-0-1,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,1,Side,/workspace/datasets/emoji-hero-vr-db-di/test_s...,[/workspace/datasets/emoji-hero-vr-db-di/test_...,Anger,Anger
2,1700479004312-2-1-1-3-0-0,1700479004312-2-1-1-3-0,1700479004312,2,1,1,3,0,0,Central,/workspace/datasets/emoji-hero-vr-db-di/test_s...,[/workspace/datasets/emoji-hero-vr-db-di/test_...,Anger,Anger
3,1700479004312-2-1-1-3-0-1,1700479004312-2-1-1-3-0,1700479004312,2,1,1,3,0,1,Side,/workspace/datasets/emoji-hero-vr-db-di/test_s...,[/workspace/datasets/emoji-hero-vr-db-di/test_...,Anger,Anger
4,1700479005401-2-1-1-4-0-0,1700479005401-2-1-1-4-0,1700479005401,2,1,1,4,0,0,Central,/workspace/datasets/emoji-hero-vr-db-di/test_s...,[/workspace/datasets/emoji-hero-vr-db-di/test_...,Anger,Anger


In [5]:
assert len(prediction_df) == 756
assert prediction_df['sample_id'].is_unique
assert prediction_df['reenactment_id'].nunique() == 378
assert set(prediction_df['camera_index']) == {0, 1}

assert (prediction_df.groupby('reenactment_id').size() == 2).all()
assert (prediction_df.groupby('reenactment_id')['camera_index'].nunique() == 2).all()
assert prediction_df['image_paths'].map(len).eq(SEQUENCE_LENGTH).all()
assert (prediction_df['emotion_dir'].map(EMOTION_TO_ID) == prediction_df['true_label_id']).all()
assert (prediction_df.groupby('true_label_id').size() == 108).all()

print('Verified 756 image-view sequences from 378 reenactments with 30 frames each.')

Verified 756 image-view sequences from 378 reenactments with 30 frames each.


### 2.1 Create and verify the deterministic shuffled test sequences

The ordered and shuffled conditions contain exactly the same 30 image files per sequence. Only the within-sequence frame order differs. The test split uses `SEED + 2`, matching the image and FEA ablation notebooks.

In [6]:
def shuffle_sequences(X, seed):
    rng = np.random.default_rng(seed)
    X_shuffled = X.copy()

    for i in range(len(X_shuffled)):
        permutation = rng.permutation(X_shuffled.shape[1])
        X_shuffled[i] = X_shuffled[i, permutation]

    return X_shuffled


X_ordered = np.array(prediction_df['image_paths'].tolist(), dtype=str)
X_shuffled = shuffle_sequences(X_ordered, TEST_SEQUENCE_SHUFFLE_SEED)

In [7]:
def verify_shuffling(X_ordered, X_shuffled):
    # The shuffled dataset must preserve the exact overall array dimensions.
    assert X_ordered.shape == X_shuffled.shape

    for i in range(len(X_ordered)):
        ordered_sequence = X_ordered[i]
        shuffled_sequence = X_shuffled[i]

        # Each sequence must still contain exactly 30 frames.
        assert len(ordered_sequence) == SEQUENCE_LENGTH
        assert len(shuffled_sequence) == SEQUENCE_LENGTH

        unique_ordered, counts_ordered = np.unique(ordered_sequence, return_counts=True)
        unique_shuffled, counts_shuffled = np.unique(shuffled_sequence, return_counts=True)

        # The shuffled sequence must contain exactly the same image paths with the same multiplicities.
        assert np.array_equal(unique_ordered, unique_shuffled) and np.array_equal(counts_ordered, counts_shuffled)

        # The frame order must actually differ after shuffling.
        assert not np.array_equal(ordered_sequence, shuffled_sequence)

    print(f'Verified {len(X_ordered)} deterministically shuffled sequences.')


verify_shuffling(X_ordered, X_shuffled)

# Repeating the operation with the same seed must reproduce exactly the same order.
assert np.array_equal(X_shuffled, shuffle_sequences(X_ordered, TEST_SEQUENCE_SHUFFLE_SEED))
print('Verified reproducibility with TEST_SEQUENCE_SHUFFLE_SEED =', TEST_SEQUENCE_SHUFFLE_SEED)

Verified 756 deterministically shuffled sequences.
Verified reproducibility with TEST_SEQUENCE_SHUFFLE_SEED = 15


## 3. Inference utilities


### 3.1 Custom layer required for loading the frozen models

The trained models contain the custom `SequenceAugment` layer. At inference time, the original layer performs no augmentation, so this compatibility implementation only casts the input to `float32`.

In [8]:
from keras import layers
from keras.saving import register_keras_serializable


@register_keras_serializable(package='seqaug')
class SequenceAugment(layers.Layer):

    def __init__(self, image_size=(224, 224), crop_scale=(0.95, 1.0), rotation_max_deg=20.0,
                 fill_mode='CONSTANT', fill_value=1.0, flip_prob=0.5, brightness_max_delta=20.0,
                 contrast_lower=0.9, contrast_upper=1.1, gamma_range=(0.9, 1.1), clip_after_color=True,
                 noise_std=4, temporal_shift_max=0, frame_drop_prob=0.05, time_mask_prob=0.05,
                 time_mask_max_frac=0.10, invert_prob=0.0, solarize_prob=0.05,
                 solarize_threshold=(120.0, 160.0), **kwargs):
        super().__init__(**kwargs)
        self.image_size = tuple(image_size)
        self.crop_scale = tuple(crop_scale)
        self.rotation_max_deg = float(rotation_max_deg)
        self.fill_mode = str(fill_mode)
        self.fill_value = float(fill_value)
        self.flip_prob = float(flip_prob)
        self.brightness_max_delta = float(brightness_max_delta)
        self.contrast_lower = float(contrast_lower)
        self.contrast_upper = float(contrast_upper)
        self.gamma_range = None if gamma_range is None else tuple(gamma_range)
        self.clip_after_color = bool(clip_after_color)
        self.noise_std = float(noise_std)
        self.temporal_shift_max = int(temporal_shift_max)
        self.frame_drop_prob = float(frame_drop_prob)
        self.time_mask_prob = float(time_mask_prob)
        self.time_mask_max_frac = float(time_mask_max_frac)
        self.invert_prob = float(invert_prob)
        self.solarize_prob = float(solarize_prob)
        self.solarize_threshold = tuple(solarize_threshold)

    def call(self, x, training=None):
        if training is True:
            raise RuntimeError('This inference-only compatibility layer must not be used for training.')
        return tf.cast(x, tf.float32)

    def get_config(self):
        config = super().get_config()
        config.update({
            'image_size': self.image_size,
            'crop_scale': self.crop_scale,
            'rotation_max_deg': self.rotation_max_deg,
            'fill_mode': self.fill_mode,
            'fill_value': self.fill_value,
            'flip_prob': self.flip_prob,
            'brightness_max_delta': self.brightness_max_delta,
            'contrast_lower': self.contrast_lower,
            'contrast_upper': self.contrast_upper,
            'gamma_range': self.gamma_range,
            'clip_after_color': self.clip_after_color,
            'noise_std': self.noise_std,
            'temporal_shift_max': self.temporal_shift_max,
            'frame_drop_prob': self.frame_drop_prob,
            'time_mask_prob': self.time_mask_prob,
            'time_mask_max_frac': self.time_mask_max_frac,
            'invert_prob': self.invert_prob,
            'solarize_prob': self.solarize_prob,
            'solarize_threshold': self.solarize_threshold
        })
        return config


CUSTOM_OBJECTS = {
    'SequenceAugment': SequenceAugment,
    'seqaug>SequenceAugment': SequenceAugment
}

### 3.2 Lazy image-sequence datasets

No sample-level dataset shuffle is used during inference. Thus, model predictions remain aligned with the rows of `prediction_df`. The ordered and mean models receive the chronological sequences; the shuffled LSTM receives the deterministic shuffled sequences prepared above.

In [9]:
def parse_image(filename: tf.Tensor) -> tf.Tensor:
    image_string = tf.io.read_file(filename)
    image = tf.io.decode_jpeg(image_string, channels=IMAGE_SIZE[2])
    return image


def load_image_sequence(image_paths: tf.Tensor) -> tf.Tensor:
    images = tf.map_fn(parse_image, image_paths, fn_output_signature=tf.uint8)
    return tf.ensure_shape(images, (SEQUENCE_LENGTH, *IMAGE_SIZE))


def create_image_dataset(image_paths, batch_size=BATCH_SIZE) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices(image_paths)
    ds = ds.map(load_image_sequence, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(1)
    return ds

### 3.3 Prediction storage

All dynamic image models use the canonical class order. Each prediction is stored together with its seven class probabilities.

In [10]:
def add_predictions(df: pd.DataFrame, prefix: str, probabilities: np.ndarray) -> np.ndarray:
    probabilities = np.asarray(probabilities)

    assert probabilities.shape == (len(df), 7)
    assert np.all(probabilities >= 0)
    assert np.all(probabilities <= 1)
    assert np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-4)

    predictions = np.argmax(probabilities, axis=1)

    df[f'{prefix}_pred_id'] = predictions
    df[f'{prefix}_pred'] = [ID_TO_EMOTION[p] for p in predictions]

    for i, emotion in enumerate(EMOTIONS):
        df[f'{prefix}_prob_{emotion.lower()}'] = probabilities[:, i]

    return predictions

## 4. Ordered LSTM

The ordered model is evaluated on the original chronological 30-frame sequences. Expected test result: **550 / 756 = 72.75%**.

In [11]:
ordered_model = keras.models.load_model(ORDERED_MODEL_PATH, custom_objects=CUSTOM_OBJECTS, compile=False, safe_mode=False)

print('Input:', ordered_model.input_shape)
print('Output:', ordered_model.output_shape)

ordered_probabilities = ordered_model.predict(create_image_dataset(X_ordered), verbose=1)
ordered_predictions = add_predictions(prediction_df, 'ordered', ordered_probabilities)

ordered_correct = np.sum(ordered_predictions == prediction_df['true_label_id'].to_numpy())
ordered_accuracy = ordered_correct / len(prediction_df)

print(f'Ordered: {ordered_correct}/756 = {ordered_accuracy:.6f}')
assert ordered_correct == 550

2026-09-18 16:58:58.999507: E tensorflow/core/util/util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


Input: (None, 30, 224, 224, 3)
Output: (None, 7)
24/24 ━━━━━━━━━━━━━━━━━━━━ 95s 2s/step   
Ordered: 550/756 = 0.727513


In [12]:
del ordered_model, ordered_probabilities
keras.backend.clear_session()
keras.mixed_precision.set_global_policy('mixed_float16')
gc.collect()

0

## 5. Shuffled LSTM

The optimized shuffled LSTM is evaluated on the same 30 frames using the deterministic test-set permutation generated with `SEED + 2`. Expected test result: **468 / 756 = 61.90%**.

In [13]:
shuffled_model = keras.models.load_model(SHUFFLED_MODEL_PATH, custom_objects=CUSTOM_OBJECTS, compile=False, safe_mode=False)

print('Input:', shuffled_model.input_shape)
print('Output:', shuffled_model.output_shape)

shuffled_probabilities = shuffled_model.predict(create_image_dataset(X_shuffled), verbose=1)
shuffled_predictions = add_predictions(prediction_df, 'shuffled', shuffled_probabilities)

shuffled_correct = np.sum(shuffled_predictions == prediction_df['true_label_id'].to_numpy())
shuffled_accuracy = shuffled_correct / len(prediction_df)

print(f'Shuffled: {shuffled_correct}/756 = {shuffled_accuracy:.6f}')
assert shuffled_correct == 468

Input: (None, 30, 224, 224, 3)
Output: (None, 7)
24/24 ━━━━━━━━━━━━━━━━━━━━ 96s 2s/step   
Shuffled: 468/756 = 0.619048


In [14]:
del shuffled_model, shuffled_probabilities
keras.backend.clear_session()
keras.mixed_precision.set_global_policy('mixed_float16')
gc.collect()

0

## 6. Mean aggregation

The optimized mean-aggregation model receives the original chronological sequence. Its temporal mean is order-invariant, so shuffling the input would not change the prediction. Expected test result: **460 / 756 = 60.85%**.

In [15]:
mean_model = keras.models.load_model(MEAN_MODEL_PATH, custom_objects=CUSTOM_OBJECTS, compile=False, safe_mode=False)

print('Input:', mean_model.input_shape)
print('Output:', mean_model.output_shape)

mean_probabilities = mean_model.predict(create_image_dataset(X_ordered), verbose=1)
mean_predictions = add_predictions(prediction_df, 'mean', mean_probabilities)

mean_correct = np.sum(mean_predictions == prediction_df['true_label_id'].to_numpy())
mean_accuracy = mean_correct / len(prediction_df)

print(f'Mean: {mean_correct}/756 = {mean_accuracy:.6f}')
assert mean_correct == 460

Input: (None, 30, 224, 224, 3)
Output: (None, 7)


I0000 00:00:1789750976.181444     248 service.cc:146] XLA service 0x77134c002d70 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1789750976.181464     248 service.cc:154]   StreamExecutor device (0): NVIDIA GeForce RTX 5090, Compute Capability 12.0
I0000 00:00:1789751000.744195     248 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


24/24 ━━━━━━━━━━━━━━━━━━━━ 127s 3s/step  
Mean: 460/756 = 0.608466


In [16]:
del mean_model, mean_probabilities
keras.backend.clear_session()
gc.collect()

0

## 7. Final validation and export

### 7.1 Integrity checks

These checks confirm the expected sample structure and complete, valid prediction probabilities for all three models.

In [17]:
prediction_df = prediction_df.sort_values(
    ['reenactment_id', 'camera_index']
).reset_index(drop=True)

In [18]:
assert len(prediction_df) == 756
assert prediction_df['sample_id'].is_unique
assert prediction_df['reenactment_id'].nunique() == 378
assert (prediction_df.groupby('reenactment_id').size() == 2).all()
assert (prediction_df.groupby('reenactment_id')['camera_index'].nunique() == 2).all()
assert (prediction_df.groupby('true_label_id').size() == 108).all()

assert prediction_df[['ordered_pred_id', 'shuffled_pred_id', 'mean_pred_id']].notna().all().all()

for prefix in ['ordered', 'shuffled', 'mean']:
    probability_columns = [f'{prefix}_prob_{emotion.lower()}' for emotion in EMOTIONS]
    assert prediction_df[probability_columns].notna().all().all()
    assert np.allclose(prediction_df[probability_columns].sum(axis=1), 1.0, atol=1e-4)

print('Final prediction table passed all integrity checks.')

Final prediction table passed all integrity checks.


### 7.2 Summarize reproduced test results

In [19]:
for model in ['ordered', 'shuffled', 'mean']:
    correct = (prediction_df[f'{model}_pred_id'] == prediction_df['true_label_id']).sum()
    print(f'{model:<10}: {correct:>3}/756 = {correct / 756:.4%}')

ordered   : 550/756 = 72.7513%
shuffled  : 468/756 = 61.9048%
mean      : 460/756 = 60.8466%


### 7.3 Export `image_test_predictions.csv`

The exported CSV contains one row per image-view sequence. Raw 30-frame path lists are omitted; identifiers, sequence-directory paths, predictions, and class probabilities are retained.

In [20]:
output_columns = [
    'sample_id',
    'reenactment_id',
    'timestamp',
    'set_id',
    'participant_id',
    'level_id',
    'emoji_id',
    'camera_index',
    'perspective',
    'true_label_id',
    'true_label',
    'image_sequence_path'
]

prediction_columns = [
    c for c in prediction_df.columns
    if c.startswith('ordered_') or c.startswith('shuffled_') or c.startswith('mean_')
]

output_df = prediction_df[output_columns + prediction_columns].copy()
output_df.to_csv('image_test_predictions.csv', index=False)

print(f'Exported {len(output_df)} rows to image_test_predictions.csv')
display(output_df.head())

Exported 756 rows to image_test_predictions.csv


,sample_id,reenactment_id,timestamp,set_id,participant_id,level_id,emoji_id,camera_index,perspective,true_label_id,...,shuffled_prob_surprise,mean_pred_id,mean_pred,mean_prob_anger,mean_prob_disgust,mean_prob_fear,mean_prob_happiness,mean_prob_neutral,mean_prob_sadness,mean_prob_surprise
0,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,Central,0,...,0.002134,0,Anger,0.553419,0.441173,0.000044,0.000001,1.509095e-10,0.005359,0.000004
1,1700478995850-2-1-1-0-0-1,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,1,Side,0,...,0.004542,1,Disgust,0.187334,0.778844,0.000102,0.000002,6.475732e-10,0.033707,0.000013
2,1700478998549-2-1-1-1-5-0,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,0,Central,5,...,0.000659,5,Sadness,0.014817,0.007378,0.000023,0.000033,1.182091e-05,0.977726,0.000012
3,1700478998549-2-1-1-1-5-1,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,1,Side,5,...,0.001852,5,Sadness,0.008396,0.008872,0.000008,0.000003,2.090306e-07,0.982719,0.000002
4,1700479001137-2-1-1-2-3-0,1700479001137-2-1-1-2-3,1700479001137,2,1,1,2,0,Central,3,...,0.007957,3,Happiness,0.000239,0.000075,0.001177,0.998240,1.545893e-05,0.000182,0.000072
